# East River streamflow and snowpack-release comparison

**WY2018–WY2026 · March 1–June 30, inclusive · no ERA5 input**

Produces one 3×3 physical-volume figure (centered 7-day means) and one
1×3 figure of annual Qsum, Qmax and CTQ curves. Numeric tables are retained
as CSV files; set `SHOW_TABLES = True` to also display them. Metrics use the
**unsmoothed daily volumes**, never the moving-average curves.

The seasonal window follows the user's latest instruction to minimize monsoon
influence. This is an **East River adaptation**, not an exact reproduction of the
paper's SWE-defined window or a source-separated estimate of snowmelt runoff.

Run all cells in order in the existing East River Python environment, or install
`numpy pandas matplotlib jupyterlab`. No workflow rerun or network request occurs.
The notebook reads existing outputs and writes into `streamflow_marjun_comparison/`
under the production output directory. It does not overwrite workflow tables.

The delivered production notebook is intentionally unexecuted. Its companion
`East_River_Streamflow_MarJun_archive_executed.ipynb` contains results from the
reviewed local archive; the live HPC directory was not accessible during development.


## Definitions and interpretation

[Lei et al. (2026), §2.2.1](https://hess.copernicus.org/articles/30/5343/2026/)
defines seasonal total runoff, maximum discharge, and CTQ as the date cumulative
runoff reaches half the seasonal total. Its ERA5-based window is replaced here
by the user-selected March–June window. The paper summarizes annual
characteristics across years; this notebook also retains every annual result.

For daily volume $V_d$ in m³/day, with each record representing one day:

\[
Q_{sum}=\sum_d V_d(1\,\mathrm{day}),\qquad
Q_{max}=\max_d(V_d/86400),\qquad
CTQ=\min\{d:\sum_{j\le d}V_j\ge 0.5\sum_jV_j\}.
\]

Qsum is reported in m³; Qmax in m³/s (maximum **daily mean**, not instantaneous
peak); CTQ as a calendar date and day of water year (Oct 1 = 1). No subdaily
interpolation is applied. CTQ is not the discharge-weighted mean date.

**Source meaning:** USGS 09112500 is observed East River discharge at Almont; iSnobal SWI and Noah-MP
QSNBOT are unrouted snowpack water-release diagnostics. Applying the same three
operations to model release does not make those series simulated outlet discharge.
The supplied slide describes SWI as snow-water input from the snowpack and
QSNBOT as water leaving its bottom. Rainfall, storage and routing effects are not removed. Basin/gauge support is
inherited from v0.8.2; equivalence to the gauge's contributing area is not newly
established here. No magnitude bias or model-skill scores are claimed.


## Reviewed sources and resolved differences

* Primary figure reference: user-supplied `SWI_QSNBOT_USGS_MarJun_3x3.ipynb`
  and the hydrograph/quantity-definition images supplied with it.
* Earlier code: `East_River_Streamflow_SWI_QSNBOT_USGS_MarJun_3x3.ipynb`,
  `streamflow_timing_normalized_7day_template.ipynb`, and
  `streamflow_fractional_timing_replot_corrected.ipynb`, found in Downloads;
  project `streamflow_USGS.py`; v0.8.2 `plot_streamflow` and `streamflow_products`.
* Documentation: both supplied v0.8.2 manuals, `PROJECT_DATA_STRUCTURE.txt`,
  current `East_River_Plotting_Requirements_JHM_AMS_v5.md`, and the code archive.
* Actual run: `East_River_Workflow_Code_v0.8.2updated.zip`, its `output/output1`
  tables, manifest, unit/time/grid reports, and source code. Relevant
  `data_access.py`, `processing.py`, and `metrics.py` match the project code.

| Difference | Treatment here |
|---|---|
| Old `/bsuscratch/...` output path | Use the user's `/bsuhome/zihaozhang/scratch/.../output/output1` path. |
| Paper SWE window; H4 March–September | Use March–June for all sources, per latest instruction; do not reuse Table 7 dates. |
| Prior physical label `m³/7 days` | Correct to `m³/day`: a 7-day **mean** does not change units. |
| Baseline smooths before slicing; March–June notebook slices first | Physical plot follows the March–June notebook, keeping July out of the endpoint. |
| Earlier request included a normalized figure | Omitted per the user's latest instruction. No normalization is computed. |
| Earlier workflow Arial/DejaVu, 1.5-pt lines, PNG only | Current Nimbus Sans, 18-pt base, 20-pt WY titles, 2.5-pt lines, normal weight, PNG+PDF. |
| Archive iSnobal root `with_spinup`; project YAML `erw_res_forcing` | Retain the archived processed values; do not rebuild from either raw path. |
| H4 ignores missing/negative values | This new seasonal table requires all 122 finite nonnegative daily values; otherwise metrics are missing with a reason. |

Model volumes are already sums of depth × fractional watershed overlap × cell area.
iSnobal SWI uses the final configured `em.nc` record, without a second hourly sum.
Noah QSNBOT is integrated over all 24 hourly records (the run reports mm/s).
USGS daily cfs is multiplied by 0.028316846592 × 86400.
The internal spelling `qsnobot_volume_m3_day` is intentionally retained.

The archive's fractional areas are 749,430,208 m² (iSnobal) and 748,679,552 m²
(Noah-MP), a 0.1002% difference. iSnobal valid area is slightly below 100% on
some days; volumes are not renormalized to fill missing pixels. Noah's daily
table has no equivalent valid-area-fraction column. These limits remain visible
in the audit rather than being corrected by plotting code.


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, ScalarFormatter
from IPython.display import display, Markdown

# PRODUCTION DEFAULT: this is the user-authorized output directory.
PRODUCTION_OUTPUT_DIR = Path(
    '/bsuhome/zihaozhang/scratch/iSnobal/'
    'East_River_Workflow_Code_v0.8.2/output/output1'
)
INPUT_MODE = 'production'  # 'archive' uses the included, explicitly identified snapshot.
NOTEBOOK_DIR = Path.cwd()  # Launch Jupyter with this notebook's folder as the working directory.
INPUT_DIR = PRODUCTION_OUTPUT_DIR if INPUT_MODE == 'production' else NOTEBOOK_DIR / 'inputs'
RESULT_DIR = (PRODUCTION_OUTPUT_DIR / 'streamflow_marjun_comparison'
              if INPUT_MODE == 'production' else NOTEBOOK_DIR / 'archive_results')
YEARS = list(range(2018, 2027))
SAVE_FIGURES = True
SAVE_TABLES = True
SHOW_FIGURES = True
SHOW_TABLES = False
PNG_DPI = 600
FONT_FILES = [NOTEBOOK_DIR / 'fonts/NimbusSans-Regular.ttf']

# Copied exactly from v0.8.2 constants.py (no workflow import/path side effects).
MODEL_COLORS = {'iSnobal': '#1F77B4', 'Noah-MP': '#D62728', 'USGS': 'black'}
SOURCES = {
    'USGS': ('usgs_discharge_daily.csv', 'discharge_m3_day', 'USGS discharge', 'observed discharge'),
    'iSnobal': ('isnobal_watershed_daily.csv', 'swi_volume_m3_day', 'iSnobal SWI', 'unrouted snowpack release'),
    'Noah-MP': ('noahmp_watershed_daily.csv', 'qsnobot_volume_m3_day', 'Noah-MP QSNBOT', 'unrouted snowpack release'),
}
if INPUT_MODE not in {'production', 'archive'}:
    raise ValueError('INPUT_MODE must be production or archive.')
if not YEARS or len(YEARS) != len(set(YEARS)) or sorted(YEARS) != YEARS:
    raise ValueError('YEARS must be nonempty, unique and increasing.')
print(f'Input mode: {INPUT_MODE}\nInput directory: {INPUT_DIR}\nNew results: {RESULT_DIR}')
print('Season: March 1–June 30 inclusive. Physical volumes only.')


## Read and audit the existing output lineage

No substitute files or downloads are selected automatically. Required manifests
and the saved streamflow product are checked against the daily source tables.
Missing files or conflicting values stop execution with their paths/names.
The full-year audit can report USGS gaps outside March–June without dropping
otherwise complete spring seasons. Content hashes are written with the results.


In [ ]:
input_hashes = {}

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_json(relative):
    path = INPUT_DIR / relative
    if not path.is_file():
        raise FileNotFoundError(f'Required workflow evidence missing: {path}')
    input_hashes[relative] = sha256(path)
    return json.loads(path.read_text(encoding='utf-8'))

def require(condition, message):
    if not condition:
        raise ValueError(message)

def read_daily(relative, required):
    path = INPUT_DIR / relative
    if not path.is_file():
        raise FileNotFoundError(f'Required daily output missing: {path}')
    input_hashes[relative] = sha256(path)
    f = pd.read_csv(path)
    require(set(['date', *required]).issubset(f.columns), f'{path}: missing required columns {required}')
    dates = pd.to_datetime(f['date'], errors='raise')
    require(dates.notna().all() and dates.dt.tz is None, f'{path}: invalid or timezone-aware dates')
    require(dates.eq(dates.dt.normalize()).all(), f'{path}: expected midnight daily date labels')
    require(not dates.duplicated().any(), f'{path}: duplicate daily dates')
    f['date'] = dates
    wy = dates.dt.year + dates.dt.month.ge(10).astype(int)
    if 'water_year' in f:
        require(pd.to_numeric(f.water_year, errors='raise').eq(wy).all(), f'{path}: inconsistent water_year')
    f['water_year'] = wy
    for col in required:
        f[col] = pd.to_numeric(f[col], errors='raise')
    return f.sort_values('date').set_index('date')

manifest = read_json('output_manifest.json')
validation = read_json('diagnostics/validation_report.json')
noah_time = read_json('diagnostics/noahmp_processing_diagnostics.json')
usgs_metadata = read_json('diagnostics/usgs_download_metadata.json')
grid_report = read_json('diagnostics/grid_diagnostics.json')
require(manifest.get('workflow_version') == '0.8.2', 'Expected workflow version 0.8.2')
require(validation['watershed']['selected_feature_count'] == 1 and
        validation['watershed']['selected_name'] == 'East River', 'Unexpected watershed selection')
require(str(usgs_metadata.get('site')) == '09112500', 'Unexpected USGS gauge')
require(noah_time['unexpected_gap_count'] == 0 and noah_time['daily_state_hour'] == 23,
        'Noah time diagnostics do not match validated processing')
require(noah_time['record_count'] == 85439 and noah_time['timestep_seconds'] == 3600,
        'Noah record count or timestep differs from reviewed production run')
start = pd.Timestamp(manifest['analysis_start']).normalize()
end = pd.Timestamp(manifest['analysis_end']).normalize()
frames = {source: read_daily('tables/' + spec[0], [spec[1]]) for source, spec in SOURCES.items()}
daily = pd.concat({s: f[SOURCES[s][1]] for s, f in frames.items()}, axis=1).sort_index()
require(daily.index.equals(pd.date_range(start, end, name='date')), 'Daily dates differ from manifest coverage')
u = frames['USGS']
require('discharge_cfs' in u, 'USGS table lacks original cfs values needed for conversion audit')
np.testing.assert_allclose(u.discharge_m3_day, u.discharge_cfs * 0.028316846592 * 86400,
                           rtol=1e-10, atol=1e-6, equal_nan=True)
product_columns = [spec[1] for spec in SOURCES.values()]
product_columns += [f'{s}_{suffix}' for s in SOURCES for suffix in ['7day_plot']]
baseline = read_daily('tables/streamflow_daily_products.csv', product_columns)
require(baseline.index.equals(daily.index), 'Saved streamflow product dates differ from daily tables')
audit_rows = []
for source, spec in SOURCES.items():
    v = daily[source]
    np.testing.assert_allclose(v, baseline[spec[1]], rtol=1e-10, atol=1e-6, equal_nan=True)
    smooth = v.rolling(7, center=True, min_periods=1).mean()
    np.testing.assert_allclose(smooth, baseline[f'{source}_7day_plot'], rtol=1e-10, atol=1e-6, equal_nan=True)
    audit_rows.append(dict(source=source, rows=len(v), first_date=v.index.min(),
                           last_date=v.index.max(), missing_daily=int(v.isna().sum()),
                           negative_daily=int(v.lt(0).sum()), baseline_agrees=True))
input_audit = pd.DataFrame(audit_rows)
display(input_audit)
print('Fractional-grid support:', grid_report.get('extra', {}))
if 'swi_valid_area_fraction' in frames['iSnobal']:
    print('Minimum iSnobal SWI valid-area fraction:', frames['iSnobal'].swi_valid_area_fraction.min())
print('All saved baseline raw, smoothed values agree with their daily inputs.')


## Seasonal calculations and separate metric tables

Each March–June window has 122 days, including in leap years. Incomplete or negative
seasons receive missing metrics and an explicit status. No zeros, interpolation,
absolute values or H4 filtering are silently substituted. A complete zero-volume
season has Qsum = Qmax = 0 and undefined CTQ.

Plots retain the prior centered seven-record mean with `min_periods=1`, after
reindexing to real calendar days. The first/last three days use shorter windows.
No normalization is computed. Smoothing does not change the metrics.

The multi-year table averages annual quantities over the **same complete,
positive-total years for all three sources** and reports the included years.
Mean CTQ is expressed as days since March 1 (March 1 = 0) to avoid leap-year
offsets; it is not computed from a pooled or averaged hydrograph.


In [ ]:
def calculate_metrics(series):
    values = series.to_numpy(dtype=float)
    missing = int((~np.isfinite(values)).sum())
    negative = int((values < 0).sum())
    result = dict(Qsum_m3=np.nan, Qmax_m3_s=np.nan, CTQ_date=pd.NaT,
                  CTQ_DOWY=np.nan, CTQ_days_since_Mar1=np.nan,
                  missing_days=missing, negative_days=negative, status='complete')
    if missing or negative:
        result['status'] = f'invalid: {missing} missing/nonfinite, {negative} negative days'
        return result
    total = float(values.sum())
    result.update(Qsum_m3=total, Qmax_m3_s=float(values.max()) / 86400)
    if total <= 0:
        result['status'] = 'zero total: CTQ undefined'
        return result
    # First daily record at/above half; exact equality selects that date.
    ctq = series.index[int(np.searchsorted(np.cumsum(values), total * 0.5, side='left'))]
    result.update(CTQ_date=ctq,
                  CTQ_DOWY=int((ctq - pd.Timestamp(ctq.year - 1, 10, 1)).days + 1),
                  CTQ_days_since_Mar1=int((ctq - pd.Timestamp(ctq.year, 3, 1)).days))
    return result

def construct_seasons(daily, years):
    plot_rows, metric_rows = [], []
    for wy in years:
        season_start, season_end = pd.Timestamp(wy, 3, 1), pd.Timestamp(wy, 6, 30)
        dates = pd.date_range(season_start, season_end, name='date')
        for source, spec in SOURCES.items():
            series = daily[source].reindex(dates)
            result = calculate_metrics(series)
            metric_rows.append(dict(water_year=wy, source=source, quantity=spec[3],
                                    season_start=season_start, season_end=season_end,
                                    expected_days=len(dates), **result))
            moving = series.rolling(7, center=True, min_periods=1).mean()
            part = pd.DataFrame(dict(date=dates, water_year=wy, source=source,
                                     daily_volume_m3_day=series.to_numpy(),
                                     physical_7day_mean_m3_day=moving.to_numpy(),
                                     daily_values_in_mean=series.rolling(7, center=True, min_periods=1).count().to_numpy()))
            plot_rows.append(part)
    return pd.DataFrame(metric_rows), pd.concat(plot_rows, ignore_index=True)

metrics, plot_data = construct_seasons(daily, YEARS)
comparison_table = metrics[['water_year', 'source', 'Qsum_m3', 'Qmax_m3_s', 'CTQ_date', 'status']].copy()
if SHOW_TABLES:
    display(Markdown('**Annual March–June characteristics — unsmoothed daily inputs**'))
    display(comparison_table.round({'Qsum_m3': 1, 'Qmax_m3_s': 3}))

good = metrics.assign(good=metrics.status.eq('complete')).groupby('water_year').good.all()
common_years = good.index[good].tolist()
summary_rows = []
for source in SOURCES:
    sub = metrics.loc[metrics.source.eq(source) & metrics.water_year.isin(common_years)]
    summary_rows.append(dict(source=source, n_years=len(sub), years=','.join(map(str, common_years)),
                             mean_annual_Qsum_m3=sub.Qsum_m3.mean(),
                             mean_annual_Qmax_m3_s=sub.Qmax_m3_s.mean(),
                             mean_annual_CTQ_days_since_Mar1=sub.CTQ_days_since_Mar1.mean()))
summary = pd.DataFrame(summary_rows)
if SHOW_TABLES:
    display(Markdown('**Across-year means — common complete seasons**'))
    display(summary.round(3))
if not metrics.status.eq('complete').all():
    warnings.warn('Some annual metrics are unavailable. Inspect status, missing_days and negative_days.')


In [ ]:
# Write tables before font checks, so numeric results remain usable independently.
if SAVE_TABLES:
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(RESULT_DIR / 'marjun_metrics_annual.csv', index=False, date_format='%Y-%m-%d')
    summary.to_csv(RESULT_DIR / 'marjun_metrics_multiyear.csv', index=False)
    plot_data.to_csv(RESULT_DIR / 'marjun_plot_data.csv', index=False, date_format='%Y-%m-%d')
    input_audit.to_csv(RESULT_DIR / 'input_audit.csv', index=False)
    html = ('<!doctype html><html><head><meta charset="utf-8"><title>East River seasonal metrics</title>'
            '<style>body{font:16px Arial,sans-serif;margin:32px}table{border-collapse:collapse}'
            'th,td{padding:8px 12px;border-bottom:1px solid #ddd;text-align:right}th{background:#eee}</style>'
            '</head><body><h1>East River: March–June seasonal characteristics</h1>'
            '<p>Unsmoothed daily volumes; model SWI/QSNBOT are unrouted snowpack release. '
            'Qsum: m³; Qmax: m³/s (daily mean); CTQ: first date at 50% cumulative volume.</p>'
            + comparison_table.to_html(index=False, float_format=lambda x: f'{x:,.3f}')
            + '<h2>Across-year means</h2>' + summary.to_html(index=False, float_format=lambda x: f'{x:,.3f}')
            + '</body></html>')
    (RESULT_DIR / 'marjun_metrics.html').write_text(html, encoding='utf-8')
    print('Saved separate metric, plot-data and audit tables to', RESULT_DIR)


## Publication figures

Nimbus Sans is required and checked without fallback. The included regular TTF
comes from [Artifex's URW base35 release 20200910](https://github.com/ArtifexSoftware/urw-base35-fonts/tree/20200910);
its license and PDF-embedding exception are included in `fonts/`.
If using only this notebook, install Nimbus Sans or set `FONT_FILES` to its file.
The primary supplied notebook is followed for solid curves, iSnobal/Noah-MP/USGS
legend order and labels, and its `nice_ymax` shared-axis calculation.
No main title or metric annotations are added. Raw gaps are recorded in the audit;
the rolling mean skips missing values as in the prior code, and the plot table
records the number of observations in every mean. Metrics do not skip missing days.


In [ ]:
def configure_style():
    for file in FONT_FILES:
        if Path(file).is_file():
            fm.fontManager.addfont(str(file))
    resolved = fm.findfont(fm.FontProperties(family='Nimbus Sans', weight='normal'), fallback_to_default=False)
    require(fm.FontProperties(fname=resolved).get_name() == 'Nimbus Sans', 'Nimbus Sans did not resolve exactly')
    mpl.rcParams.update({'font.family': 'Nimbus Sans', 'font.size': 18, 'font.weight': 'normal',
                         'axes.titlesize': 20, 'axes.titleweight': 'normal', 'axes.labelsize': 18,
                         'axes.labelweight': 'normal', 'xtick.labelsize': 18, 'ytick.labelsize': 18,
                         'legend.fontsize': 18, 'figure.labelsize': 18, 'figure.labelweight': 'normal',
                         'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.unicode_minus': True})
    print('Verified figure font:', resolved)
    return resolved

def nice_ymax(values):
    # Exact helper from the user-supplied SWI_QSNBOT_USGS_MarJun_3x3.ipynb.
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return 1.0
    top = float(np.nanmax(arr)) * 1.05
    if top <= 0:
        return 1.0
    raw_step = top / 6.0
    exponent = 10 ** np.floor(np.log10(raw_step))
    candidates = exponent * np.array([1.0, 2.0, 2.5, 5.0, 10.0])
    step = float(candidates[candidates >= raw_step][0])
    return float(np.ceil(top / step) * step)


def plot_comparison(frame):
    col = 'physical_7day_mean_m3_day'
    nrows = math.ceil(len(YEARS) / 3)
    fig, axes = plt.subplots(nrows, 3, figsize=(21, 5.5 * nrows), sharey=True, squeeze=False)
    finite = frame[col].to_numpy(dtype=float)
    finite = finite[np.isfinite(finite)]
    require(len(finite) > 0, f'No finite values to plot: {col}')
    upper = nice_ymax(finite)
    lower = min(0.0, float(finite.min()) * 1.05)
    plot_order = ['iSnobal', 'Noah-MP', 'USGS']
    styles = {source: '-' for source in plot_order}
    for panel_index, (ax, wy) in enumerate(zip(axes.flat, YEARS)):
        for source in plot_order:
            sub = frame.loc[frame.water_year.eq(wy) & frame.source.eq(source)]
            ax.plot(sub.date, sub[col], color=MODEL_COLORS[source], lw=2.5,
                    ls=styles[source], label=SOURCES[source][2])
        ax.set_title(f'WY {wy}', fontsize=20, fontweight='normal', loc='center', y=1.06, pad=8)
        ax.set_title(f'({chr(97 + panel_index)})', fontsize=18, fontweight='normal', loc='left', y=1.06, pad=8)
        ax.set_xlim(pd.Timestamp(wy, 3, 1), pd.Timestamp(wy, 6, 30))
        ax.set_ylim(lower, upper)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['left', 'bottom']].set_linewidth(1)
        ax.tick_params(axis='both', which='major', width=1, length=5)
        formatter = ScalarFormatter(useMathText=False)
        formatter.set_powerlimits((0, 0))
        ax.yaxis.set_major_formatter(formatter)
    for ax in axes.flat[len(YEARS):]:
        ax.set_visible(False)
    handles = [Line2D([], [], color=MODEL_COLORS[s], lw=2.5, ls=styles[s], label=s) for s in plot_order]
    axes.flat[0].legend(handles=handles, loc='upper left', frameon=True, facecolor='white',
                        edgecolor='0.4', framealpha=0.9, fancybox=False, borderpad=0.5, handlelength=2.2)
    fig.supxlabel('Month', y=0.095)
    fig.supylabel('7-day mean daily volume (m³/day)', x=0.015)
    fig.subplots_adjust(top=0.95, bottom=0.15, left=0.07, hspace=0.30, wspace=0.16)
    return fig

def export_pair(fig, stem):
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    paths = [RESULT_DIR / f'{stem}.png', RESULT_DIR / f'{stem}.pdf']
    fig.savefig(paths[0], dpi=PNG_DPI, bbox_inches='tight', pad_inches=0.05, facecolor='white')
    fig.savefig(paths[1], bbox_inches='tight', pad_inches=0.05, facecolor='white')
    require(all(p.is_file() and p.stat().st_size > 0 for p in paths), 'Dual figure export failed')
    return paths

font_path = configure_style()
figure_paths = []
fig = plot_comparison(plot_data)
if SAVE_FIGURES:
    figure_paths.extend(export_pair(fig, 'streamflow_physical_7day_marjun_3x3'))
if SHOW_FIGURES:
    plt.show()
plt.close(fig)
print('Saved figure files:', *figure_paths, sep='\n')


## Annual metric curves: 1 × 3

Panels (a)–(c) show Qsum, Qmax and CTQ across water years, respectively.
Each point uses the March–June **unsmoothed** daily data already calculated above.
Connecting lines join annual values; no additional smoothing is applied.
Qsum is displayed in million m³, Qmax in m³/s, and CTQ in **day of water year
(October 1 = 1)**. Leap days are included in that timing coordinate; the CSV
retains exact dates and days since March 1 as well.

The figure follows the current project protocol: verified Nimbus Sans, 18-pt
base text, 20-pt normal-weight panel titles, 2.5-pt source-colored lines, boxed
legend, no main title, and 600-dpi PNG plus embedded-font PDF. Different point
markers provide an additional source cue. The first-panel legend is at upper
right to keep the large WY2019 values visible. Undefined metrics remain gaps.


In [ ]:
def plot_metric_curves(metric_frame):
    panel_specs = [
        ('Qsum_m3', 1e6, 'Qsum', 'Seasonal volume (million m³)'),
        ('Qmax_m3_s', 1.0, 'Qmax', 'Peak daily-mean flow (m³/s)'),
        ('CTQ_DOWY', 1.0, 'CTQ', 'Timing (day of water year)'),
    ]
    plot_order = ['iSnobal', 'Noah-MP', 'USGS']
    markers = {'iSnobal': 'o', 'Noah-MP': 's', 'USGS': '^'}
    fig, axes = plt.subplots(1, 3, figsize=(21, 5.5), sharex=True, squeeze=False)
    for panel_index, (ax, spec) in enumerate(zip(axes.flat, panel_specs)):
        column, divisor, title, ylabel = spec
        for source in plot_order:
            series = (metric_frame.loc[metric_frame.source.eq(source)]
                      .set_index('water_year')[column].reindex(YEARS))
            ax.plot(YEARS, series / divisor, color=MODEL_COLORS[source],
                    linewidth=2.5, linestyle='-', marker=markers[source],
                    markersize=6, markeredgewidth=1, label=source)
        ax.set_title(title, fontsize=20, fontweight='normal', loc='center', y=1.06, pad=8)
        ax.set_title(f'({chr(97 + panel_index)})', fontsize=18,
                     fontweight='normal', loc='left', y=1.06, pad=8)
        ax.set_ylabel(ylabel)
        ax.set_xticks(YEARS[::2])
        ax.set_xlim(min(YEARS) - 0.3, max(YEARS) + 0.3)
        values = metric_frame.loc[metric_frame.water_year.isin(YEARS), column].to_numpy(dtype=float) / divisor
        values = values[np.isfinite(values)]
        if len(values):
            if column == 'CTQ_DOWY':
                ax.set_ylim(math.floor((values.min() - 8) / 10) * 10,
                            math.ceil((values.max() + 8) / 10) * 10)
            else:
                headroom = 1.35 if panel_index == 0 else 1.08
                ax.set_ylim(0, float(MaxNLocator(nbins=6).tick_values(0, max(values.max() * headroom, 1))[-1]))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=column == 'CTQ_DOWY'))
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['left', 'bottom']].set_linewidth(1.0)
        ax.tick_params(axis='both', which='major', width=1.0, length=5)
    axes.flat[0].legend(loc='upper right', ncol=1, frameon=True, facecolor='white',
                        edgecolor='0.4', framealpha=0.9, fancybox=False,
                        borderpad=0.5, handlelength=2.2)
    fig.supxlabel('Water year', y=0.025)
    fig.subplots_adjust(top=0.84, bottom=0.19, left=0.065, right=0.99, wspace=0.30)
    return fig

metric_fig = plot_metric_curves(metrics)
if SAVE_FIGURES:
    figure_paths.extend(export_pair(metric_fig, 'marjun_metrics_curves_1x3'))
if SHOW_FIGURES:
    plt.show()
plt.close(metric_fig)
print('Metric curves use the 27 annual source/year results; numeric CSV files remain available.')


In [ ]:
provenance = {
    'input_mode': INPUT_MODE, 'input_dir': str(INPUT_DIR),
    'production_output_dir': str(PRODUCTION_OUTPUT_DIR), 'result_dir': str(RESULT_DIR),
    'workflow_version': manifest['workflow_version'], 'input_sha256': input_hashes,
    'season': 'March 1–June 30 inclusive; fixed by user; no ERA5 input',
    'metric_input': 'unsmoothed daily basin volumes; no clipping or gap filling',
    'Qsum_units': 'm3', 'Qmax_units': 'm3/s; maximum daily mean',
    'CTQ_rule': 'first calendar day at or above 50 percent cumulative seasonal volume',
    'model_quantity': 'unrouted snowpack release, not routed outlet discharge',
    'normalization': 'none, per user instruction', 'physical_smoothing': 'season-clipped centered 7-day mean; min_periods=1',
    'years': YEARS, 'multiyear_common_years': common_years,
    'font_path': str(font_path), 'font_sha256': sha256(font_path), 'png_dpi': PNG_DPI,
    'figure_paths': [str(p) for p in figure_paths],
    'metric_curve_axes': {'Qsum': 'million m3', 'Qmax': 'm3/s', 'CTQ': 'day of water year; Oct 1 = 1'},
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'matplotlib': mpl.__version__},
    'paper': 'https://doi.org/10.5194/hess-30-5343-2026',
}
if SAVE_TABLES or SAVE_FIGURES:
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    (RESULT_DIR / 'run_provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
display(Markdown('**Finished.** The hydrograph and 1×3 annual metric curves are saved separately. '
                 'Check the status column and run_provenance.json before interpreting results.'))
